In [ ]:
#1 AE

import numpy as np
# NORMAL data only (training)
normal_data = data_scaled[data_scaled["isolation_forest_anomaly"] == 1]
X_ae_train = normal_data[scaled_features].values
print("AE training shape:", X_ae_train.shape)

# ALL data (scoring)
X_ae_all = data_scaled[scaled_features].values
print("AE evaluation shape:", X_ae_all.shape)



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

# Use the scaled versions of your numeric features
scaled_features = [f"{col}_scaled" for col in numeric_features]

# Autoencoder input matrix
# Use Isolation Forest NORMAL transactions only
normal_data = data_scaled[data_scaled["isolation_forest_anomaly"] == 1]

scaled_features = [f"{col}_scaled" for col in numeric_features]
X_ae = normal_data[scaled_features].values

print("Autoencoder training data shape:", X_ae.shape)



In [ ]:
X_train, X_val = train_test_split(
    X_ae,
    test_size=0.1,
    random_state=42,
    shuffle=True
)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

input_dim = X_ae.shape[1]

autoencoder = Sequential([
    Input(shape=(input_dim,)),         # <-- defines input properly
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),        # bottleneck
    Dense(16, activation="relu"),
    Dense(input_dim, activation="linear")
])

autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()


In [ ]:
history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=50,
    batch_size=32,
    shuffle=True,
    verbose=1
)


In [ ]:
# Reconstruct all samples
X_all = data_scaled[scaled_features].values
reconstructions_all = autoencoder.predict(X_all)

# Reconstruction error per transaction
reconstruction_error_all = np.mean(
    np.square(X_all - reconstructions_all),
    axis=1
)

data_scaled["ae_error"] = reconstruction_error_all
print(data_scaled["ae_error"].describe())


In [ ]:
# 95th percentile threshold from AE reconstruction error
threshold = data_scaled["ae_error"].quantile(0.95)
print("Autoencoder threshold (95th percentile):", threshold)

# -1 = anomaly, 1 = normal
data_scaled["ae_anomaly"] = np.where(
    data_scaled["ae_error"] > threshold,
    -1,
    1
)

ae_anomaly_count = (data_scaled["ae_anomaly"] == -1).sum()
ae_normal_count  = (data_scaled["ae_anomaly"] == 1).sum()

print("Autoencoder anomalies:", ae_anomaly_count)
print("Autoencoder normal   :", ae_normal_count)



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.hist(data_scaled["ae_error"], bins=50)
plt.axvline(
    data_scaled["ae_error"].quantile(0.95),
    linestyle="--",
    label="95th percentile"
)
plt.title("Autoencoder Reconstruction Error Distribution")
plt.xlabel("Reconstruction error")
plt.ylabel("Count")
plt.legend()
plt.show()
